# repairs

> what a reading gets wrong, put right: a table of fixes before assembly and one after, unnumbered runs, numbers borrowed from a second reading, chapters cut by a plan, pages that are not the text, sūtras

In [ ]:
#| default_exp repairs

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from difflib import SequenceMatcher
from lekhaka.verses import vnum, blocks, kind, _bk

These work on pages of blocks, `[{page, blocks}]` (or `items`), before `assemble`, and change them in place. Each change is noted in `probs` for a proofreader. They are what vedicreader's books needed again and again: the fix tables of the Bṛhat Saṃhitā, the Muhūrta Cintāmaṇi and the Nārada Saṃhitā, the numbering runs of the Phaladīpikā, the second readings of the Jātaka Pārijāta, the heading plans of the Grahacintāmaṇi. `fix_chapters` and `sutras` work on the chapters `assemble` returns.

## A table of fixes

`apply_fixes` applies rows `(page, item, kind, read, printed, why)`. `item` is a block's index on its page. `kind` is `fix` (replace `read` with `printed` in the block's text, once; `fix_all` every time), `doubt` (the same, but only where `witness` says another print agrees; else noted and kept), `slip` (the print's own slip, kept and noted), `number` (the block's number is `printed`), `drop` (the block goes), or `missing` (`printed` is a whole block, put in after `item`; -1 puts it first).

In [ ]:
#| export
def apply_fixes(pages, rows, witness:dict=None, probs:list=None) -> list:
    "Pages with a table of fixes applied: `(page, item, kind, read, printed, why)` rows."
    probs = [] if probs is None else probs
    by, adds = {p['page']: p for p in pages}, []
    for pg, i, kind, a, b, why in rows:
        if (p := by.get(pg)) is None: continue
        bl = blocks(p)
        if kind == 'missing': adds.append((pg, i, b, why)); continue
        it = bl[i] if isinstance(i, int) and 0 <= i < len(bl) else None
        if it is None: probs.append(f'{pg}: block {i} is gone from the reading; {kind} {a} → {b} not applied'); continue
        if kind == 'drop': it['drop'] = True; probs.append(f"{pg}: `{(it.get('text') or '')[:30]}` dropped ({why})"); continue
        if kind == 'number': it['n'] = b; probs.append(f'{pg}: verse number read {a}, printed {b}'); continue
        if a not in (it.get('text') or ''): probs.append(f'{pg}: `{a}` not in block {i}; the {kind} to `{b}` not applied'); continue
        w = (witness or {}).get((pg, i, a))
        if kind in ('fix', 'fix_all') or (kind == 'doubt' and w == 'agrees'):
            m = it['text'].count(a) if kind == 'fix_all' else 1
            it['text'] = it['text'].replace(a, b, m)
            probs.append(f'{pg}: read `{a}`, printed `{b}`' + (f', {m} times' if m > 1 else '') + (f' ({why})' if why else '') + (' [a doubt the witness settles]' if kind == 'doubt' else ''))
        elif kind == 'doubt': probs.append(f'{pg}: doubt: `{a}` kept; perhaps `{b}` ({why})')
        elif kind == 'slip': probs.append(f"{pg}: the print's slip, kept: `{a}` ({why})")
    for _, (pg, after, blk, why) in sorted(enumerate(adds), key=lambda x: (-(x[1][1] if isinstance(x[1][1], int) else 999), -x[0])):
        blocks(by[pg]).insert(after + 1 if isinstance(after, int) and after >= 0 else 0, dict(blk, added=True))
        probs.append(f"{pg}: {blk.get('kind')} {blk.get('n') or ''} missing from the reading, put in: {(blk.get('text') or '')[:40]}")
    for p in pages: p[_bk(p)] = [b for b in blocks(p) if not b.get('drop')]
    return pages

In [ ]:
_pg = lambda: [dict(page='p1', blocks=[dict(kind='verse', n=1, text='रामः वनं गच्छति ॥१॥'), dict(kind='verse', n=3, text='सीता ॥३॥'), dict(kind='prose', text='a note')])]
_p = []
_r = apply_fixes(_pg(), [('p1', 0, 'fix', 'गच्छति', 'गच्छत्', 'the print'), ('p1', 1, 'number', 3, 2, ''), ('p1', 2, 'drop', None, None, 'a footnote'),
                         ('p1', 1, 'missing', None, dict(kind='verse', n=3, text='लक्ष्मणः ॥३॥'), 'at the foot'), ('p1', 0, 'doubt', 'रामः', 'राम', 'faint')], probs=_p)
test_eq([(b['n'], b['text']) for b in _r[0]['blocks']], [(1, 'रामः वनं गच्छत् ॥१॥'), (2, 'सीता ॥३॥'), (3, 'लक्ष्मणः ॥३॥')])
test_eq(len(_p), 5); assert any('doubt' in x for x in _p)
test_eq(apply_fixes(_pg(), [('p1', 0, 'doubt', 'रामः', 'राम', '')], witness={('p1', 0, 'रामः'): 'agrees'})[0]['blocks'][0]['text'][:3], 'राम')
_p = []; test_eq(apply_fixes([dict(page=0, blocks=[dict(kind='verse', text='क क ख')])], [(0, 0, 'fix_all', 'क', 'ग', '')], probs=_p)[0]['blocks'][0]['text'], 'ग ग ख')
test_eq(_p, ['0: read `क`, printed `ग`, 2 times'])

## Numbers

`number_runs` numbers a run of unnumbered verses that fits exactly between two numbered ones (a, k verses, then a+k+1); a colophon, or a heading matching `reset`, starts the count again. `borrow_numbers` gives an unnumbered verse the number a second reading prints on the verse most like it, at a similarity of `at` or more, when no verse on the page has it already.

In [ ]:
#| export
def number_runs(pages, probs:list=None, reset:str=r'ध्याय') -> list:
    "Unnumbered verses numbered where they fit exactly between two numbered ones."
    probs = [] if probs is None else probs
    a, run = 0, []
    for p in pages:
        for b in blocks(p):
            k = kind(b)
            if k == 'colophon' or (k == 'heading' and reset and re.search(reset, b.get('text') or '')): a, run = 0, []; continue
            if k != 'verse': continue
            if (n := vnum(b.get('n'))) is None: run.append((p['page'], b)); continue
            if run and n - a - 1 == len(run):
                for j, (pg, x) in enumerate(run, 1): x['n'] = a + j; probs.append(f'{pg}: verse printed without its number taken as {a + j}, between {a} and {n}')
            a, run = n, []
    return pages

def borrow_numbers(pages, other, probs:list=None, at:float=.6) -> list:
    "Unnumbered verses given the number `other`, a second reading of the same pages as pages, prints on the most similar verse."
    probs = [] if probs is None else probs
    theirs = {p['page']: [(n, b.get('text') or '') for b in blocks(p) if kind(b) == 'verse' and (n := vnum(b.get('n'))) is not None] for p in other}
    for p in pages:
        if not (th := theirs.get(p['page'])): continue
        have = {vnum(b.get('n')) for b in blocks(p) if kind(b) == 'verse'}
        for b in blocks(p):
            if kind(b) != 'verse' or b.get('n') is not None: continue
            sc, n = max((SequenceMatcher(None, b.get('text') or '', t).ratio(), n) for n, t in th)
            if sc >= at and n not in have: b['n'] = n; have.add(n); probs.append(f"{p['page']}: verse {n}'s number, lost in this reading, taken from the other ({sc:.2f})")
    return pages

In [ ]:
_ps = [dict(page=0, blocks=[dict(kind='verse', n=1, text='a'), dict(kind='verse', text='b'), dict(kind='verse', text='c'), dict(kind='verse', n=4, text='d'),
                             dict(kind='colophon', text='इति'), dict(kind='verse', text='e'), dict(kind='verse', n=3, text='f')])]
test_eq([b.get('n') for b in number_runs(_ps)[0]['blocks']], [1, 2, 3, 4, None, None, 3])   # after the colophon one verse cannot fill 1..2
_mine = [dict(page=0, blocks=[dict(kind='verse', text='धर्मक्षेत्रे कुरुक्षेत्रे'), dict(kind='verse', n=2, text='दृष्ट्वा तु')])]
_theirs = [dict(page=0, blocks=[dict(kind='verse', n='१', text='धर्मक्षेत्रे कुरुक्षेत्रे समवेता'), dict(kind='verse', n='२', text='दृष्ट्वा तु')])]
test_eq([b['n'] for b in borrow_numbers(_mine, _theirs)[0]['blocks']], [1, 2])

In [ ]:
_it = [dict(page=0, items=[dict(kind='verse', n=1, text='a'), dict(kind='verse', text='b'), dict(kind='head', text='द्वितीयोऽध्यायः'), dict(kind='verse', text='c'), dict(kind='verse', n=2, text='d')])]
test_eq([b.get('n') for b in number_runs(_it)[0]['items']], [1, None, None, 1, 2])   # `items` read as blocks, a `head` as a heading
test_eq([b['n'] for b in borrow_numbers([dict(page=0, items=[dict(kind='verse', text='धर्मक्षेत्रे कुरुक्षेत्रे')])], _theirs)[0]['items']], [1])
test_eq(len(apply_fixes([dict(page=0, items=[dict(kind='verse', text='x')])], [(0, 0, 'drop', None, None, '')])[0]['items']), 0)

## Chapters by plan

`cut_by_plan` marks where chapters begin when a print's numbering does not say: `plan` lists `(page, item)` pairs, where `item` is a block index or the start of a heading's text. A `start` block goes in before each, and `assemble` closes the chapter there. Each cut is noted in `probs`; a third value in a row is why.

In [ ]:
#| export
def cut_by_plan(pages, plan, probs:list=None) -> list:
    "Pages with a chapter-opening `start` block at each `(page, item, why)` of `plan`, each cut noted; `item` is an index or a heading's first words."
    probs = [] if probs is None else probs
    by = {p['page']: p for p in pages}
    for pg, item, *why in plan:
        if (p := by.get(pg)) is None: probs.append(f'{pg}: the plan names a page the reading lacks'); continue
        bl = blocks(p)
        i = item if isinstance(item, int) else next((j for j, b in enumerate(bl) if (b.get('text') or '').startswith(item)), None)
        if i is None or not 0 <= i < len(bl): probs.append(f'{pg}: the plan\'s chapter start {item!r} is not on the page'); continue
        b, y = bl[i], f' ({why[0]})' if why and why[0] else ''
        w = f'verse {m}' if kind(b) == 'verse' and (m := vnum(b.get('n'))) is not None else f"`{(b.get('text') or '')[:30]}`"
        bl.insert(i, dict(kind='start')); probs.append(f'{pg}: a chapter starts at {w}, by the plan{y}')
    return pages

In [ ]:
from lekhaka.verses import assemble
_pl = [dict(page=0, blocks=[dict(kind='heading', text='கடவுள் வணக்கம்'), dict(kind='verse', n=1, text='x'), dict(kind='heading', text='வாரநயம்'), dict(kind='verse', n=2, text='y')])]
_p = []; _c, _ = assemble(cut_by_plan(_pl, [(0, 'வாரநயம்')], _p)); test_eq(_p, ['0: a chapter starts at `வாரநயம்`, by the plan'])
test_eq([(c['heading'], c['colophon'], [v['n'] for v in c['verses']]) for c in _c], [('கடவுள் வணக்கம்', None, [1]), ('வாரநயம்', None, [2])])
_p = []; cut_by_plan(_pl, [(5, 0), (0, 'அது')], _p); test_eq(len(_p), 2)
_p = []; cut_by_plan([dict(page='p2', items=[dict(kind='verse', n='२', text='x')])], [('p2', 0, 'the leaf before it lost')], _p)
test_eq(_p, ['p2: a chapter starts at verse 2, by the plan (the leaf before it lost)'])

## Pages that are not the text

`drop_pages` drops the verses on each `(first, last)` range of pages: an appendix of another work's verses after the text (the Phaladīpikā's Jātakābharaṇa verses after 26.48).

In [ ]:
#| export
def drop_pages(pages, ranges, probs:list=None) -> list:
    "Pages with the verses dropped on each `(first, last)` range of pages that are not the text."
    probs = [] if probs is None else probs
    for a, b in ranges:
        for p in pages:
            if a <= p['page'] <= b and any(kind(x) == 'verse' for x in blocks(p)):
                p[_bk(p)] = [x for x in blocks(p) if kind(x) != 'verse']; probs.append(f"{p['page']}: verses dropped, not the text")
    return pages

In [ ]:
_p = []
_d = drop_pages([dict(page=k, items=[dict(kind='heading', text='h'), dict(kind='verse', n=k, text='v')]) for k in range(1, 5)], [(2, 3)], _p)
test_eq([len(p['items']) for p in _d], [2, 1, 1, 2]); test_eq(_p, ['2: verses dropped, not the text', '3: verses dropped, not the text'])

## After assembly

`fix_chapters` applies a table of fixes to the chapters `assemble` returns, keyed by chapter and verse: rows `(chapter, verse, kind, read, printed, why)`. `chapter` is a chapter's place from 1, or what `key(i, chapter)` returns (an edition's own chapter numbers, a `(part, no)`); `None` is every chapter that has the verse. Rows apply in order, so a row after a `start` counts the chapters as split. `kind` is

- `fix`: replace `read` with `printed`, once, in the verse's text; `fix_all`: every time it occurs; `slip`, `doubt`, `note`: noted and kept, as `apply_fixes` notes them (a `note` is its `why` alone); `number`: the verse's number is `printed`; `drop`: the verse goes;
- `missing`: a verse the reading lacks, read by hand from another copy: `printed` is its text, or a dict `{text, page, …}`, and it goes in its place by number;
- `recut`: where `mend` cannot see the boundary (a four-line verse whose last line has no daṇḍa), verse `verse` starts at line `printed` of it and the verse before joined;
- `start`: a chapter starts at verse `verse`, where a lost leaf takes a chapter's end and the next one's start with it. When a chapter holds two verses of that number, `printed` picks one: the page it is on, or which one, 1 the first.

`sutras` makes each verse of a sūtra text one line: wrapped lines joined, a hyphen at a break dropped, `close` (॥) put at the end where the reading lost it.

In [ ]:
#| export
_KINDS = ('fix', 'fix_all', 'slip', 'doubt', 'note', 'number', 'drop', 'missing', 'recut', 'start')
def _at(vs, n): return next((i for i, v in enumerate(vs) if v['n'] == n), None)

def fix_chapters(chs, rows, probs:list=None, key=None) -> list:
    "Chapters with a table of fixes applied: `(chapter, verse, kind, read, printed, why)` rows; `chapter` None is every chapter with the verse."
    probs, key = [] if probs is None else probs, key or (lambda i, c: i)
    for ch, n, k, a, b, why in rows:
        if k not in _KINDS: raise ValueError(f'{k}: not a kind of fix')
        w, y = f'{ch}.{n}' if ch is not None else f'{n}', f' ({why})' if why else ''
        cs = [c for i, c in enumerate(chs, 1) if ch is None or key(i, c) == ch]
        if k == 'note': probs.append(f'{w}: {why}'); continue
        if not cs: probs.append(f'{w}: no chapter {ch}; the {k} not applied'); continue
        if k == 'missing':
            vs = cs[0]['verses']
            if _at(vs, n) is not None: probs.append(f'{w}: verse {n} is in the reading; the missing verse not put in'); continue
            v = dict(n=n, text=b, page=None, heading=None) if isinstance(b, str) else dict(dict(page=None, heading=None), **b, n=n)
            vs.insert(next((i for i, x in enumerate(vs) if x['n'] > n), len(vs)), dict(v, added=True))
            probs.append(f"{w}: verse {n} missing from the reading, put in: {v['text'][:40]}{y}"); continue
        if k == 'start':
            hs = [(c, i) for c in cs for i, v in enumerate(c['verses']) if v['n'] == n and (not isinstance(b, str) or v['page'] == b)]
            hs = hs if isinstance(b, str) else hs[b - 1:b] if isinstance(b, int) else [(c, _at(c['verses'], n)) for c in cs]
            c, i = next(((c, i) for c, i in hs if i), (None, None))
            if c is None: probs.append(f'{w}: no verse {n} after the first of a chapter' + ('' if b is None else f' ({b})') + '; the start not applied'); continue
            vs, j = c['verses'], next(j for j, x in enumerate(chs) if x is c)
            chs[j:j + 1] = [dict(c, verses=vs[:i], colophon=None), dict(c, heading=None, verses=vs[i:])]
            probs.append(f"{vs[i]['page']}: a chapter starts at verse {w}{y}"); continue
        if k == 'recut':
            vs = cs[0]['verses']
            if not (i := _at(vs, n)): probs.append(f'{w}: no verse before {n} to recut it from'); continue
            ls = (vs[i - 1]['text'] + '\n' + vs[i]['text']).split('\n')
            vs[i - 1], vs[i] = dict(vs[i - 1], text='\n'.join(ls[:b])), dict(vs[i], text='\n'.join(ls[b:]))
            probs.append(f"{vs[i]['page']}: {w} begins at line {b + 1} of {n - 1}-{n} as read{y}"); continue
        hits = [(c, i) for c in cs if (i := _at(c['verses'], n)) is not None]
        if not hits: probs.append(f'{w}: verse {n} is gone from the reading; {k} {a} → {b} not applied'); continue
        for c, i in hits:
            v = c['verses'][i]; pg = f"{key(next(j for j, x in enumerate(chs, 1) if x is c), c)}.{n} ({v['page']})"
            if k == 'drop': del c['verses'][i]; probs.append(f"{pg}: `{v['text'][:30]}` dropped{y}")
            elif k == 'number': c['verses'][i] = dict(v, n=b); probs.append(f'{pg}: verse number read {a}, printed {b}')
            elif k == 'slip': probs.append(f"{pg}: the print's slip, kept: `{a}`{y}")
            elif k == 'doubt': probs.append(f'{pg}: doubt: `{a}` kept; perhaps `{b}`{y}')
            elif a not in v['text']: probs.append(f'{pg}: `{a}` not in the verse; the fix to `{b}` not applied')
            else:
                m = v['text'].count(a) if k == 'fix_all' else 1
                c['verses'][i] = dict(v, text=v['text'].replace(a, b, m)); probs.append(f'{pg}: read `{a}`, printed `{b}`' + (f', {m} times' if m > 1 else '') + y)
    return chs

def sutras(chs, probs:list=None, close:str='॥', key=None) -> list:
    "Chapters of sūtras, each one line: wrapped lines joined, a hyphen at a break dropped, closed by `close` where the reading lost it."
    probs, key = [] if probs is None else probs, key or (lambda i, c: i)
    for i, c in enumerate(chs, 1):
        out = []
        for v in c['verses']:
            u = re.sub(r'-\s*\n\s*', '', v['text']).replace('\n', ' ').strip()
            if not u.endswith(close): u = re.sub(r'[।|]$', '', u) + close; probs.append(f"{key(i, c)}.{v['n']} ({v['page']}): closing {close} supplied")
            out.append(dict(v, text=u))
        c['verses'] = out
    return chs

In [ ]:
V = lambda n, t, pg=1: dict(n=n, text=t, page=pg, heading=None)
_ch = lambda: [dict(heading='h1', colophon='c1', verses=[V(1, 'रामः गच्छति ॥१॥'), V(2, 'सीता ॥२॥'), V(4, 'क\nख\nग ॥४॥'), V(5, 'घ ॥५॥')]), dict(heading='h2', colophon='c2', verses=[V(1, 'रामः ॥१॥')])]
_p = []
_c = fix_chapters(_ch(), [(1, 1, 'fix', 'गच्छति', 'गच्छत्', 'typed'), (None, 1, 'fix', 'रामः', 'राम', ''), (1, 3, 'missing', None, 'लक्ष्मणः ॥३॥', 'from the 1905 print'),
                          (1, 5, 'recut', None, 2, ''), (1, 2, 'slip', 'सीता', None, 'so printed'), (2, 9, 'fix', 'x', 'y', ''), (1, 1, 'note', None, None, 'a lacuna'),
                          (1, 4, 'start', None, None, 'a leaf lost'), (2, 5, 'drop', None, None, 'a quotation')], _p)
test_eq([(c['heading'], c['colophon'], [(v['n'], v['text']) for v in c['verses']]) for c in _c],
        [('h1', None, [(1, 'राम गच्छत् ॥१॥'), (2, 'सीता ॥२॥'), (3, 'लक्ष्मणः ॥३॥')]), (None, 'c1', [(4, 'क\nख')]), ('h2', 'c2', [(1, 'राम ॥१॥')])])
test_eq(_p[:3], ['1.1 (1): read `गच्छति`, printed `गच्छत्` (typed)', '1.1 (1): read `रामः`, printed `राम`', '2.1 (1): read `रामः`, printed `राम`'])
test_eq(len(_p), 10); assert '2.9: verse 9 is gone from the reading; fix x → y not applied' in _p
test_eq([v['n'] for v in fix_chapters(_ch(), [((1, 'x'), 2, 'drop', None, None, '')], key=lambda i, c: (i, 'x'))[0]['verses']], [1, 4, 5])
test_fail(lambda: fix_chapters(_ch(), [(1, 1, 'mend', 'a', 'b', '')]))
_p = []; _c = fix_chapters([dict(heading=None, colophon=None, verses=[V(1, 'क क ॥१॥')])], [(1, 1, 'fix_all', 'क', 'ग', 'twice')], _p)
test_eq((_c[0]['verses'][0]['text'], _p), ('ग ग ॥१॥', ['1.1 (1): read `क`, printed `ग`, 2 times (twice)']))
_two = lambda: [dict(heading='h7', colophon='c8', verses=[V(1, 'a'), V(2, 'b'), V(25, 'y', 'p0107a'), V(2, 'z', 'p0108a'), V(3, 'w', 'p0108a')])]
_p = []; _c = fix_chapters(_two(), [(None, 2, 'start', None, 'p0108a', 'the leaf before it lost')], _p)   # two verses 2 in one chapter: the page picks
test_eq([[v['n'] for v in c['verses']] for c in _c], [[1, 2, 25], [2, 3]]); test_eq(_p, ['p0108a: a chapter starts at verse 2 (the leaf before it lost)'])
test_eq([[v['n'] for v in c['verses']] for c in fix_chapters(_two(), [(1, 2, 'start', None, 2, '')])], [[1, 2, 25], [2, 3]])   # or which one, from 1
test_eq([[v['n'] for v in c['verses']] for c in fix_chapters(_two(), [(1, 2, 'start', None, None, '')])], [[1], [2, 25, 2, 3]])   # else the first
_p = []; fix_chapters(_two(), [(1, 2, 'start', None, 'p0999a', '')], _p); test_eq(_p, ['1.2: no verse 2 after the first of a chapter (p0999a); the start not applied'])
_p = []
test_eq(sutras([dict(verses=[V(1, 'अथातो धर्म-\nजिज्ञासा ।'), V(2, 'चोदनालक्षणोऽर्थो धर्मः ॥')])], _p)[0]['verses'][0]['text'], 'अथातो धर्मजिज्ञासा ॥'); test_eq(_p, ['1.1 (1): closing ॥ supplied'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()